# صدى — تشغيل الخادم على Kaggle (GPU مجاني) + نفق عام (Cloudflare Tunnel)

يشغّل هذا الدفتر خادم FastAPI الخاص بمشروع **صدى** باستخدام نموذج التفريغ الصوتي العربي المحلي
(`CohereLabs/cohere-transcribe-arabic-07-2026`) على GPU مجاني من Kaggle، ثم يفتح رابطًا عامًا عبر
Cloudflare Tunnel (وضع Quick Tunnel — بدون حساب أو توكن) لتتصل به الواجهة الأمامية (المستضافة على
GitHub Pages أو أي مكان آخر).

## قبل التشغيل
1. **Settings → Accelerator → GPU T4 x2** (أو أي GPU متاح).
2. **Settings → Internet → On** (مطلوب لتنزيل الحزم والاتصال بـ OpenAI/Hugging Face/Cloudflare).
3. **Add-ons → Secrets** أضف:
   - `OPENAI_API_KEY` — لخطوة التلخيص عبر LangChain/ChatGPT.
   - `HF_TOKEN` — توكن Hugging Face **بصلاحية قراءة**. الموديل مقيّد (gated)؛ لازم تطلب الوصول
     إليه من صفحته على Hugging Face أولاً (Request access) وإلا ستحصل على خطأ 403 عند التحميل.
   
   (لا حاجة لأي سرّ خاص بالنفق — Cloudflare Quick Tunnel لا يتطلب تسجيل حساب ولا توكن،
   على عكس ngrok سابقًا.)

## بعد التشغيل
انسخ الرابط العام (`https://xxxx.trycloudflare.com`) الذي تطبعه آخر خلية، والصقه في زر **"رابط الخادم"**
أعلى صفحة الواجهة. الرابط يتغيّر في كل جلسة جديدة — لازم تحدّثه كل مرة تعيد فيها تشغيل هذا الدفتر.


In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: لا يوجد GPU — فعّل Settings > Accelerator، وإلا سيكون التفريغ بطيئًا جدًا.")

CUDA available: True
GPU: Tesla T4


In [2]:
REPO_URL = "https://github.com/hessenali383/SADA" 

import os
if not os.path.isdir("SADA"):
    get_ipython().system('git clone -q {REPO_URL}')
%cd SADA
!ls

/kaggle/working/SADA
backend  frontend  kaggle  LICENSE  README.md


In [3]:
%%capture
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q -r backend/requirements.txt -r backend/requirements-kaggle.txt

In [4]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def get_secret(name, required=True):
    try:
        return secrets.get_secret(name)
    except Exception:
        if required:
            raise RuntimeError(f"السر '{name}' غير موجود — أضِفه من Add-ons > Secrets قبل المتابعة.")
        return ""

os.environ["HF_TOKEN"] = get_secret("HF_TOKEN")
os.environ["GEMINI_API_KEY"] = get_secret("GEMINI_API_KEY")

os.environ["STT_PROVIDER"] = "local"
os.environ["LOCAL_ASR_MODEL"] = "CohereLabs/cohere-transcribe-arabic-07-2026"

os.environ.setdefault("GEMINI_MODEL", "gemini-3.6-flash")
os.environ.setdefault("MONTHLY_TOKEN_BUDGET", "1000000")
os.environ.setdefault("ALLOWED_ORIGINS", "*")

print("تم ضبط متغيرات البيئة")

تم ضبط متغيرات البيئة


In [5]:
import subprocess, time, requests

log_file = open("/kaggle/working/uvicorn.log", "w")
server = subprocess.Popen(
    ["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log_file, stderr=subprocess.STDOUT,
)

print("جارٍ تحميل النموذج وبدء الخادم (قد يستغرق دقيقة أو دقيقتين أول مرة)...")
for _ in range(120):
    try:
        r = requests.get("http://127.0.0.1:8000/api/status", timeout=2)
        if r.status_code == 200:
            print("الخادم جاهز:", r.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print("لم يستجب الخادم بعد — راجع /kaggle/working/uvicorn.log للتفاصيل.")

جارٍ تحميل النموذج وبدء الخادم (قد يستغرق دقيقة أو دقيقتين أول مرة)...
الخادم جاهز: {'month': '2026-08', 'budget': 1000000, 'used': 0, 'remaining': 1000000, 'percent_remaining': 100}


In [6]:
import subprocess, time, re

# cloudflared ships as a static binary, not a pip package — download it once per session
get_ipython().system(
    'wget -q -O /usr/local/bin/cloudflared "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64" '
    '&& chmod +x /usr/local/bin/cloudflared'
)


tunnel_log = open("/kaggle/working/cloudflared.log", "w")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url = None
print("جارٍ فتح نفق Cloudflare...")
for _ in range(60):
    time.sleep(2)
    log_text = open("/kaggle/working/cloudflared.log", encoding="utf-8", errors="ignore").read()
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break

print("=" * 60)
if public_url:
    print("رابط الخادم العام (الصقه في زر 'رابط الخادم' في الواجهة):")
    print(public_url)
else:
    print("لم يظهر الرابط بعد — راجع /kaggle/working/cloudflared.log للتفاصيل.")
print("=" * 60)


جارٍ فتح نفق Cloudflare...
رابط الخادم العام (الصقه في زر 'رابط الخادم' في الواجهة):
https://gratuit-movement-lightweight-periodically.trycloudflare.com
